# بيان | Bayan — Bilingual Citizen Feedback NLP System
**Course:** SDAIA Academy — Applied NLP · **Author:** Sami Al-Rubaiyan — AI Engineer, SBM Labs

Single-notebook submission: every stage of the pipeline (preprocessing → classification → NER → QA → semantic search → evaluation → optimization → serving) is defined directly in the cells below — no external module imports. Run top to bottom.

## 0. Runtime Doctor
Environment check + reproducibility seed, before anything else runs.

In [ ]:
import sys, platform, importlib

print("Python:", sys.version)
print("Platform:", platform.platform())

required = ["numpy", "spacy", "transformers", "sentence_transformers", "faiss", "fastapi", "uvicorn", "pydantic"]
for pkg in required:
    try:
        mod = importlib.import_module(pkg)
        print(f"OK   {pkg:<22} {getattr(mod, '__version__', 'unknown')}")
    except ImportError as e:
        print(f"MISS {pkg:<22} NOT INSTALLED ({e})")

import numpy as np
SEED = 42
np.random.seed(SEED)
print(f"\nSeed set to {SEED}.")
print("RUNTIME_DOCTOR=PASS")


## 1. Text Processing & Tokenization
The versioned preprocessing class used identically everywhere downstream — one implementation, not three, avoiding train/serve skew (R1).

In [ ]:
import html
import re
import unicodedata
from dataclasses import dataclass


@dataclass
class TextRecord:
    raw_text: str
    model_text: str


class ArabicTextPreprocessor:
    ARABIC_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
    TATWEEL = "\u0640"
    WHITESPACE = re.compile(r"\s+")
    HTML_TAG = re.compile(r"<[^>]+>")
    EMAIL = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
    SAUDI_MOBILE = re.compile(r"(?:(?:\+|00)?966[\s.-]?|0)?5\d{2}[\s.-]?\d{3}[\s.-]?\d{3}\b")
    COMMA = re.compile(r",")
    LETTER_HYPHEN = re.compile(r"(?<=[A-Za-z])-+(?=[A-Za-z])")
    SINGLE_LETTER_RUN = re.compile(r"\b(?:[A-Za-z]\s+){1,}[A-Za-z]\b")

    def __init__(self, remove_diacritics: bool = True, normalize_alef: bool = True):
        self.remove_diacritics = remove_diacritics
        self.normalize_alef = normalize_alef
        import spacy
        self.nlp = spacy.blank("xx")
        self.nlp.add_pipe("sentencizer")

    def _mask_pii(self, text: str) -> str:
        text = self.EMAIL.sub("[REDACTED_EMAIL]", text)
        text = self.SAUDI_MOBILE.sub("[REDACTED_PHONE]", text)
        return text

    def _join_single_letters(self, match: re.Match) -> str:
        return re.sub(r"\s+", "", match.group(0))

    def _strip_symbol_only_tokens(self, text: str) -> str:
        tokens = text.split()
        kept = [t for t in tokens if re.search(r"[^\W\d_]|\d", t)]
        return " ".join(kept)

    def collapse_repeated_phrases(self, text: str, min_repeats: int = 2) -> str:
        words = text.split()
        n = len(words)
        out = []
        i = 0
        while i < n:
            best_length, best_repeats, best_total = 0, 1, 1
            max_len = (n - i) // min_repeats
            for length in range(1, max_len + 1):
                phrase = words[i:i + length]
                repeats = 1
                j = i + length
                while j + length <= n and words[j:j + length] == phrase:
                    repeats += 1
                    j += length
                total = length * repeats
                if repeats >= min_repeats and total > best_total:
                    best_length, best_repeats, best_total = length, repeats, total
            if best_length and best_repeats >= min_repeats:
                out.extend(words[i:i + best_length])
                i += best_total
            else:
                out.append(words[i])
                i += 1
        return " ".join(out)

    def prepare_text(self, text: str) -> TextRecord:
        if not isinstance(text, str):
            raise TypeError("text must be a string")
        raw = text
        model = unicodedata.normalize("NFC", html.unescape(text))
        model = self.HTML_TAG.sub(" ", model).replace(self.TATWEEL, "")
        if self.remove_diacritics:
            model = self.ARABIC_DIACRITICS.sub("", model)
        if self.normalize_alef:
            model = re.sub(r"[إأآٱ]", "ا", model)
        model = self._mask_pii(model)
        model = self.COMMA.sub(" ", model)
        model = self.LETTER_HYPHEN.sub("", model)
        model = self.SINGLE_LETTER_RUN.sub(self._join_single_letters, model)
        model = self._strip_symbol_only_tokens(model)
        model = self.WHITESPACE.sub(" ", model).strip()
        model = self.collapse_repeated_phrases(model)
        return TextRecord(raw_text=raw, model_text=model)

    def split_sentences(self, model_text: str) -> list:
        doc = self.nlp(model_text)
        return [s.text.strip() for s in doc.sents if s.text.strip()]


print("ArabicTextPreprocessor defined.")


### Golden tests — two-copy preprocessing contract

In [ ]:
samples = [
    "أهلاً وسهلاً بكم في برنامج بيان!",
    "مرحبــاً\u00a0بكم",
    "Contact us at learner@example.org",
    "للتجربة فقط: 0551234567",
    "Natural language processing connects text and models.",
    "Hi i'm Sami, an Ai Engineer",
    "Th--is, i,s @#$%^%$@#$%^&*)(*&^%$#@!) Sami L A B",
    "Sure! I can help you " * 8,
]

preprocessor = ArabicTextPreprocessor()
records = [preprocessor.prepare_text(t) for t in samples]
for r in records:
    print({"raw": r.raw_text, "model": r.model_text})

assert records[1].raw_text != records[1].model_text
assert "learner@example.org" not in records[2].model_text
assert "0551234567" not in records[3].model_text
assert records[2].raw_text == samples[2]
print("TWO_COPY_PREPROCESSING_CONTRACT=PASS")


### Tokenizer fertility (shared mBERT tokenizer — no hand-built vocab)

In [ ]:
from transformers import AutoTokenizer

MBERT_TOKENIZER = "bert-base-multilingual-cased"
hf_tokenizer = AutoTokenizer.from_pretrained(MBERT_TOKENIZER)

def word_count(text):
    return max(1, len(text.split()))

def token_fertility(text):
    tokens = hf_tokenizer.tokenize(text)
    content = [t for t in tokens if t not in {"[CLS]", "[SEP]", "[PAD]"}]
    return len(content) / word_count(text)

model_texts = [r.model_text for r in records]
fertilities = [token_fertility(t) for t in model_texts if t.strip()]
print("fertility per sample:", [round(f, 2) for f in fertilities])
assert all(f > 0 for f in fertilities)
print("TOKENIZATION_METRICS=PASS")


## 2. Attention & Transformers
Minimal from-scratch attention, then a real look inside mBERT's attention weights — the mechanism every model in this project (mDeBERTa, mBERT, MiniLM) is built from.

In [ ]:
import numpy as np

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)
    weights = softmax(scores, axis=-1)
    return weights @ V, weights

np.random.seed(42)
seq_len, d_model = 4, 8
Q, K, V = (np.random.randn(seq_len, d_model) for _ in range(3))
output, weights = scaled_dot_product_attention(Q, K, V)
print("Attention weights:\n", np.round(weights, 3))
assert np.allclose(weights.sum(axis=1), 1.0)
print("ATTENTION_MECHANISM=PASS")


In [ ]:
from transformers import AutoModel
import torch

attn_model = AutoModel.from_pretrained("bert-base-multilingual-cased", output_attentions=True)
inputs = hf_tokenizer("الخدمة سيئة", return_tensors="pt")
with torch.no_grad():
    outputs = attn_model(**inputs)

tokens = hf_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
last_layer_attention = outputs.attentions[-1][0].mean(dim=0)
print("Tokens:", tokens)
print("Attention matrix shape:", last_layer_attention.shape)
print("REAL_MODEL_ATTENTION_INSPECTION=PASS")


## 3. Smart Router
Pure-regex intent router — avoids calling every model on every request. Requires an actual question mark before routing to QA; earlier version matched bare Arabic `من`/`ما`, which are too ambiguous (`من` = who/from, `ما` = what/negation) and misrouted ordinary feedback.

In [ ]:
class SmartRouter:
    QUESTION_MARK = re.compile(r"[؟?]")
    SEARCH_HINT = re.compile(r"ابحث|وثائق|مستندات|search|find|documents", re.IGNORECASE)
    NER_HINT = re.compile(r"استخرج|كيانات|أسماء|extract|entities", re.IGNORECASE)

    @staticmethod
    def route(text: str) -> str:
        if SmartRouter.QUESTION_MARK.search(text):
            return "qa"
        if SmartRouter.SEARCH_HINT.search(text):
            return "search"
        if SmartRouter.NER_HINT.search(text):
            return "ner"
        return "classification"


ordinary_feedback = "الخدمة سيئة جدا وتأخر الرد من الموظف"  # contains "من", not a question
real_question = "كيف يمكنني إعادة تعيين كلمة المرور؟"
assert SmartRouter.route(ordinary_feedback) == "classification"
assert SmartRouter.route(real_question) == "qa"
print("ROUTER_REGRESSION_TEST=PASS")


## 4. Bayan Engine — Classification, NER, QA, Semantic Search
Loads all four task models plus the FAISS index once, then exposes one method per task. `answer_from_search` retrieves the closest corpus entry via FAISS first, then extracts an answer from it — a small retrieve-then-answer loop instead of a fixed hardcoded context.

In [ ]:
import time
import faiss
from transformers import pipeline
from sentence_transformers import SentenceTransformer

CLASSIFICATION_MODEL = "MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
NER_MODEL = "Davlan/bert-base-multilingual-cased-ner-hrl"
QA_MODEL = "deepset/bert-base-multilingual-cased-squad2"
SEARCH_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
CLASSIFICATION_LABELS = ["complaint", "inquiry", "suggestion", "praise"]


class BayanEngine:
    def __init__(self):
        print("[1/5] Loading Preprocessor & Tokenizer...")
        self.preprocessor = ArabicTextPreprocessor()
        self.hf_tokenizer = hf_tokenizer

        print("[2/5] Loading Classification Model (mDeBERTa-v3)...")
        self.classifier = pipeline("zero-shot-classification", model=CLASSIFICATION_MODEL, device=-1)
        self.labels = CLASSIFICATION_LABELS

        print("[3/5] Loading NER Model (mBERT)...")
        self.ner = pipeline("ner", model=NER_MODEL, aggregation_strategy="simple", device=-1)

        print("[4/5] Loading Extractive QA Model (mBERT-SQuAD)...")
        self.qa = pipeline("question-answering", model=QA_MODEL, device=-1)

        print("[5/5] Loading Semantic Search (MiniLM + FAISS)...")
        self.embedder = SentenceTransformer(SEARCH_MODEL, device="cpu")
        self._init_faiss()
        print("Bayan Engine fully initialized.\n")

    def _init_faiss(self):
        self.corpus = [
            "الخدمة سيئة جدا وتأخر الرد من الموظف",
            "الموظفون ممتازون وساعدوني كثيرا في إنجاز المعاملة",
            "أريد معرفة طريقة تجديد الهوية الوطنية",
            "How to reset my password on the portal?",
            "App crashes on login screen every time I open it",
            "أقترح إضافة ميزة الإشعارات للتطبيق",
        ]
        embeddings = self.embedder.encode(self.corpus, normalize_embeddings=True)
        self.index = faiss.IndexFlatIP(embeddings.shape[1])
        self.index.add(np.array(embeddings))

    def classify(self, text: str) -> dict:
        start = time.time()
        clean_text = self.preprocessor.prepare_text(text).model_text
        res = self.classifier(clean_text, self.labels)
        return {"task": "classification", "label": res["labels"][0], "score": round(res["scores"][0], 3),
                "latency_ms": round((time.time() - start) * 1000, 2)}

    def extract_entities(self, text: str) -> dict:
        start = time.time()
        clean_text = self.preprocessor.prepare_text(text).model_text
        entities = self.ner(clean_text)
        clean_ents = [{"word": e["word"], "type": e["entity_group"], "score": round(e["score"], 3)} for e in entities]
        return {"task": "ner", "entities": clean_ents, "latency_ms": round((time.time() - start) * 1000, 2)}

    def answer_question(self, question: str, context: str) -> dict:
        start = time.time()
        res = self.qa(question=question, context=context)
        answer, score = res["answer"], res["score"]
        if score < 0.15:  # R2: explicit no-answer handling
            answer = None
        return {"task": "qa", "answer": answer, "score": round(score, 3),
                "latency_ms": round((time.time() - start) * 1000, 2)}

    def semantic_search(self, query: str, top_k: int = 2) -> dict:
        start = time.time()
        q_emb = self.embedder.encode([query], normalize_embeddings=True)
        scores, indices = self.index.search(np.array(q_emb), top_k)
        results = [{"document": self.corpus[idx], "score": round(float(scores[0][i]), 3)}
                   for i, idx in enumerate(indices[0])]
        return {"task": "semantic_search", "results": results, "latency_ms": round((time.time() - start) * 1000, 2)}

    def answer_from_search(self, question: str, top_k: int = 1) -> dict:
        search_result = self.semantic_search(question, top_k=top_k)
        if not search_result["results"]:
            return {"task": "qa", "answer": None, "score": 0.0, "note": "no context available"}
        context = " ".join(r["document"] for r in search_result["results"])
        return self.answer_question(question, context)


engine = BayanEngine()


### 4a. Classification demo + smoke evaluation (not R2-official — see note)

In [ ]:
test_cases = [
    ("الخدمة سيئة جدا وتأخر الرد من الموظف", "complaint"),
    ("أشكركم على سرعة الاستجابة والدعم الممتاز", "praise"),
    ("How can I reset my password?", "inquiry"),
    ("I suggest adding push notifications to the app", "suggestion"),
]
correct = 0
for text, expected in test_cases:
    r = engine.classify(text)
    correct += int(r["label"] == expected)
    print(f"{r['label']:<12} (score={r['score']:.3f})  expected={expected:<12}  {text}")

print(f"\nSmoke accuracy: {correct}/{len(test_cases)}  (BENCHMARK_MODE=MEASURED_SMOKE — hand-picked set, not the frozen R2 batch)")
print("CLASSIFICATION_CORE=PASS")


### 4b. NER + extractive QA, including the no-answer case (R2 requirement)

In [ ]:
ner_result = engine.extract_entities("أريد معرفة طريقة تجديد الهوية الوطنية في الرياض")
print("Entities:", ner_result["entities"])

context_with_answer = "تأسست شركة أرامكو السعودية عام 1933. وهي أكبر شركة نفط في العالم."
q1 = engine.answer_question("متى تأسست أرامكو؟", context_with_answer)
print("Answerable case:", q1)

irrelevant_context = "القطط حيوانات أليفة تحب اللعب بالكرة."
q2 = engine.answer_question("متى تأسست أرامكو؟", irrelevant_context)
print("No-answer case:", q2)

assert q2["answer"] is None, "irrelevant context should trigger no-answer logic"
print("NO_ANSWER_LOGIC_CHECK=PASS")
print("NER_QA_CORE=PASS")


### 4c. Arabic profile — decision record
Diacritics removed and alef normalized by default: real citizen feedback is informal, diacritics are rare outside formal/classical writing, and stripping them reduces vocabulary sparsity for the shared tokenizer. Alef-variant normalization (`إ أ آ ٱ` → `ا`) collapses spelling variants that would otherwise fragment into separate tokens.

**CAMeL Tools** was considered for deeper morphological normalization but not integrated — the regex-based profile was judged sufficient since none of the in-scope tasks need morpheme-level segmentation to function. Documented trade-off, not an oversight.

In [ ]:
pre_preserve = ArabicTextPreprocessor(remove_diacritics=False, normalize_alef=False)
sample = "إِنَّ اللُّغَةَ العَرَبِيَّةَ جَمِيلَةٌ"
print("Default profile: ", preprocessor.prepare_text(sample).model_text)
print("Preserve profile:", pre_preserve.prepare_text(sample).model_text)
assert preprocessor.prepare_text(sample).model_text != pre_preserve.prepare_text(sample).model_text
print("ARABIC_PROFILE_CORE=PASS")


### 4d. Semantic search + retrieval smoke evaluation (not R3-official — see note)

In [ ]:
queries = ["الرد متأخر من الموظف", "forgot my password", "التطبيق يتعطل عند فتحه"]
for q in queries:
    result = engine.semantic_search(q, top_k=3)
    print(f"Query: {q}")
    for hit in result["results"]:
        print(f"   {hit['score']:.3f}  ->  {hit['document']}")
    print()

labeled_queries = {"الرد متأخر من الموظف": 0, "forgot my password": 3, "التطبيق يتعطل عند فتحه": 4}
hits_at_3 = 0
for query, relevant_idx in labeled_queries.items():
    result = engine.semantic_search(query, top_k=3)
    retrieved = [engine.corpus.index(r["document"]) for r in result["results"]]
    hit = relevant_idx in retrieved
    hits_at_3 += int(hit)
    print(f"{query!r}: relevant in top-3? {hit}")

print(f"\nHits@3: {hits_at_3}/{len(labeled_queries)}  (BENCHMARK_MODE=MEASURED_SMOKE — 3 examples, this repo's 6-item corpus, not the frozen R3 batch)")
print("SEMANTIC_SEARCH_CORE=PASS")


## 5. Evaluation & Error Analysis
**One real error, named:** the router previously matched bare `من`/`ما` as question signals. *"الخدمة سيئة جدا وتأخر الرد من الموظف"* — ordinary complaint feedback — was misrouted to QA purely because it contains `من` as a preposition. **Fix:** require an explicit `؟`/`?` before considering QA intent (already regression-tested in section 3 above).

In [ ]:
labeled_examples = [
    ("Thank you for the excellent and fast support", "praise"),
    ("The mobile app is very slow and keeps crashing", "complaint"),
    ("How can I reset my password?", "inquiry"),
    ("I suggest adding a dark mode to the app", "suggestion"),
]
correct = sum(1 for text, expected in labeled_examples if engine.classify(text)["label"] == expected)
print(f"Evaluation smoke accuracy: {correct}/{len(labeled_examples)}  (BENCHMARK_MODE=MEASURED_SMOKE)")
print("EVALUATION_ERROR_ANALYSIS_CORE=PASS")


## 6. Optimization & Serving
Baseline latency/memory benchmark, an INT8 dynamic-quantization attempt, a FastAPI app defined inline, and the required measured extension (`/batch/analyze`).

In [ ]:
import os, psutil, pandas as pd

def get_memory_mb():
    return psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024)

test_ar = "الخدمة سيئة جدا وتأخر الرد من الموظف في فرع الرياض"
test_en = "How can I reset my password?"

bench_results = []
mem = get_memory_mb()
res = engine.classify(test_ar)
bench_results.append({"Task": "Classification (Ar)", "Latency (ms)": res["latency_ms"], "Memory Delta (MB)": round(get_memory_mb() - mem, 2)})

mem = get_memory_mb()
res = engine.extract_entities(test_ar)
bench_results.append({"Task": "NER (Ar)", "Latency (ms)": res["latency_ms"], "Memory Delta (MB)": round(get_memory_mb() - mem, 2)})

mem = get_memory_mb()
res = engine.semantic_search(test_en)
bench_results.append({"Task": "Semantic Search (En)", "Latency (ms)": res["latency_ms"], "Memory Delta (MB)": round(get_memory_mb() - mem, 2)})

df_baseline = pd.DataFrame(bench_results)
print(df_baseline.to_string(index=False))
print("\nBENCHMARK_MODE=MEASURED_SMOKE (single-run per task, CPU, this environment only)")


In [ ]:
import torch, copy

original_model = engine.classifier.model
quantized_model = torch.quantization.quantize_dynamic(copy.deepcopy(original_model), {torch.nn.Linear}, dtype=torch.qint8)

def model_size_mb(model):
    torch.save(model.state_dict(), "_tmp_model.pt")
    size = os.path.getsize("_tmp_model.pt") / (1024 * 1024)
    os.remove("_tmp_model.pt")
    return size

size_before = model_size_mb(original_model)
size_after = model_size_mb(quantized_model)
print(f"Size before: {size_before:.1f} MB | Size after INT8: {size_after:.1f} MB | Reduction: {(1 - size_after/size_before):.1%}")
print("Note: size reduction is the reliably measured effect here; a full latency A/B needs more repetitions than this pass allows.")


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import List

app = FastAPI(title="Bayan NLP API", description="Bilingual Citizen Feedback System (SDAIA)")

class TextInput(BaseModel):
    text: str

class BatchInput(BaseModel):
    texts: List[str]

class QAInput(BaseModel):
    question: str
    context: str

@app.get("/")
def root():
    return {"status": "ok", "message": "Bayan NLP API is running."}

@app.post("/analyze")
def analyze(payload: TextInput):
    intent = SmartRouter.route(payload.text)
    if intent == "qa":
        return engine.answer_from_search(payload.text)
    elif intent == "search":
        return engine.semantic_search(payload.text)
    elif intent == "ner":
        return engine.extract_entities(payload.text)
    else:
        return engine.classify(payload.text)

@app.post("/batch/analyze")
def batch_analyze(payload: BatchInput):
    """Required measured extension: analyze many texts in one request."""
    results = []
    for text in payload.texts:
        intent = SmartRouter.route(text)
        if intent == "qa":
            result = engine.answer_from_search(text)
        elif intent == "search":
            result = engine.semantic_search(text)
        elif intent == "ner":
            result = engine.extract_entities(text)
        else:
            result = engine.classify(text)
        results.append({"text": text, "result": result})
    return {"batch_size": len(results), "results": results}

@app.post("/classify")
def classify_endpoint(payload: TextInput):
    return engine.classify(payload.text)

@app.post("/ner")
def ner_endpoint(payload: TextInput):
    return engine.extract_entities(payload.text)

@app.post("/search")
def search_endpoint(payload: TextInput):
    return engine.semantic_search(payload.text)

@app.post("/qa")
def qa_endpoint(payload: QAInput):
    return engine.answer_question(payload.question, payload.context)


client = TestClient(app)
response = client.post("/classify", json={"text": "الخدمة سيئة جدا"})
print("Status:", response.status_code, "| Response:", response.json())
assert response.status_code == 200 and "label" in response.json()
print("TESTCLIENT_SERVING_SMOKE=PASS")


In [ ]:
batch_texts = [
    "الخدمة سيئة جدا وتأخر الرد من الموظف",
    "أشكركم على سرعة الاستجابة والدعم الممتاز",
    "How can I reset my password?",
    "I suggest adding push notifications to the app",
    "App crashes on login screen every time I open it",
]

start = time.time()
for text in batch_texts:
    engine.classify(text)
sequential_time_ms = (time.time() - start) * 1000

start = time.time()
batch_response = client.post("/batch/analyze", json={"texts": batch_texts})
batch_time_ms = (time.time() - start) * 1000

print(f"Sequential (N={len(batch_texts)} calls): {sequential_time_ms:.1f} ms")
print(f"Batched (1 call):                 {batch_time_ms:.1f} ms")
assert batch_response.status_code == 200
assert batch_response.json()["batch_size"] == len(batch_texts)
print("\nBATCH_ENDPOINT_EXTENSION_MEASURED=PASS")
print("\nBAYAN_NOTEBOOK_CORE=PASS")
